# Comparaison des tarifs 2026 : outil d'aide à la décision

Ce notebook compare les tarifs possibles pour le parking de Lausanne en 2026, afin de choisir la recommandation de
la partie F. **Il ne fait pas partie du rendu** : c'est un outil de discussion pour le groupe. Les chiffres
retenus dans le rapport devront ensuite être produits par `analysis.py` (variable `RECO_FEE`).

**Comment ça marche** : le notebook lance `analysis.py` et réutilise ses résultats (élasticités, fonctions de
scénario, données 2025). Il ne modifie aucun fichier.

**Tarifs comparés** : 4 CHF (situation 2025), 5, 6, 8 et 10 CHF, les scénarios de la partie D.

**Quatre critères** (ceux que la partie F doit mettre en balance) :
1. **Efficacité** : combien d'automobilistes et de jours de voiture en moins ?
2. **Recette** : combien rapporte le parking ?
3. **Fiabilité** : les différentes variantes de prévision sont-elles d'accord ?
4. **Charge** : quelle part du revenu pour ceux qui continuent à conduire, en particulier les groupes sans
   alternative ?

**Deux modèles de prévision, pour deux usages** (comme dans le notebook principal) :
- **effets globaux** (sections 1 et 2) : élasticités **nettes de Sion** (scénario central de la partie D), avec
  trois variantes ;
- **effets par groupe** (sections 3 et 4) : élasticités **par classe d'accès TP** (comme la projection de la partie
  E), car pour savoir qui paie, il faut savoir qui reste automobiliste. Ce modèle prévoit un peu moins de départs
  que le scénario central.

In [1]:
%%capture
%run analysis.py

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30, "display.width", 200)

FEES = [4, 5, 6, 8, 10]
FIVE_DAYS = 5   # working days per week, to convert car days per employee into cars parked on an average day

cen = scenario(e_did[0], e_did[1], FEES)              # central: net of Sion (Table C3)
raw = scenario(e_raw[0], e_raw[1], FEES)              # Lausanne before-after
lo = scenario(ci_did[0, 0], ci_did[0, 1], FEES)       # strong response (CI bounds)
hi = scenario(ci_did[1, 0], ci_did[1, 1], FEES)       # weak response (CI bounds)
cls = scenario_by_class(FEES)                         # elasticities by transit access class
lin = cen.copy()                                      # linear in log(fee), LPM (c)
lin["S"] = S4 + a_c * np.log(lin.fee / FEE_2025)
lin["T"] = lin.S * lin.D
lin["rev"] = lin.fee * lin["T"] * WEEKS * N_LAU
print("Elasticities, central (net of Sion): extensive", round(e_did[0], 3), "intensive", round(e_did[1], 3))
print("Elasticities by transit class (ext, int):", {c: (round(a, 3), round(b, 3)) for c, (a, b) in eps_cls.items()})

Elasticities, central (net of Sion): extensive -0.368 intensive -0.102
Elasticities by transit class (ext, int): {'Good': (np.float64(-0.517), np.float64(-0.125)), 'Medium': (np.float64(-0.288), np.float64(-0.059)), 'Poor': (np.float64(-0.025), np.float64(-0.052))}


## 1. Effets globaux (scénario central)

**Colonnes** :
- *Drivers* : automobilistes attendus sur 800 employés ; *vs 2025* : écart avec les 351 de 2025.
- *Fewer drivers per extra CHF* : automobilistes en moins pour chaque franc ajouté depuis le tarif précédent du
  tableau (rendement marginal).
- *Cars parked per day* : voitures garées un jour moyen = jours de voiture par employé × 800 / 5. Utile au
  responsable mobilité pour juger la pression sur le parking.
- *Revenue* : recette annuelle, en milliers de CHF.

In [3]:
t1 = pd.DataFrame({
    "Fee (CHF)": cen.fee,
    "Drivers": (cen.S * N_LAU).round(0).astype(int),
    "vs 2025": (cen.S * N_LAU - S4 * N_LAU).round(0).astype(int),
    "Car share (%)": (100 * cen.S).round(1),
    "Change (pp)": (100 * (cen.S - S4)).round(1),
    "Days per driver": cen.D.round(2),
    "Car days per employee": cen["T"].round(2),
    "Car days change (%)": (100 * (cen["T"] / cen["T"].iloc[0] - 1)).round(1),
    "Cars parked per day": (cen["T"] * N_LAU / FIVE_DAYS).round(0).astype(int),
    "Revenue (kCHF)": (cen.rev / 1000).round(0).astype(int),
    "Revenue change (%)": (100 * (cen.rev / cen.rev.iloc[0] - 1)).round(1),
})
drv = cen.S * N_LAU
t1["Fewer drivers per extra CHF"] = [np.nan] + [round(-(drv.iloc[k] - drv.iloc[k - 1]) / (FEES[k] - FEES[k - 1]), 1)
                                               for k in range(1, len(FEES))]
t1

,Fee (CHF),Drivers,vs 2025,Car share (%),Change (pp),Days per driver,Car days per employee,Car days change (%),Cars parked per day,Revenue (kCHF),Revenue change (%),Fewer drivers per extra CHF
0,4,351,0,43.9,0.0,3.45,1.51,0.0,242,223,0.0,NaN
1,5,323,-28,40.4,-3.5,3.37,1.36,-10.0,218,251,12.5,27.7
2,6,302,-49,37.8,-6.1,3.31,1.25,-17.4,200,276,23.9,21.0
3,8,272,-79,34.0,-9.9,3.22,1.09,-27.8,175,322,44.3,15.2
4,10,250,-101,31.3,-12.6,3.14,0.98,-35.0,157,362,62.4,10.7


## 2. Fiabilité : les variantes sont-elles d'accord ?

Part voiture (%) et recette (milliers de CHF) selon cinq façons de prévoir. *Spread* = écart entre la variante la
plus haute et la plus basse : plus il est grand, moins la prévision est fiable. Rappel : la plage crédible va de 4 à
8 CHF (un doublement, comme celui observé) ; 10 CHF est une extrapolation.

In [4]:
variants = {"Central (net of Sion)": cen, "CI: strong response": lo, "CI: weak response": hi,
            "Before-after": raw, "By transit class": cls, "Linear LPM (c)": lin}
share = pd.DataFrame({k: (100 * v.S).round(1).values for k, v in variants.items()}, index=FEES)
share["Spread (pp)"] = share.max(axis=1) - share.min(axis=1)
rev = pd.DataFrame({k: (v.rev / 1000).round(0).astype(int).values for k, v in variants.items() if k != "Linear LPM (c)"},
                   index=FEES)
rev["Spread (kCHF)"] = rev.max(axis=1) - rev.min(axis=1)
share.index.name = rev.index.name = "Fee (CHF)"
display(Markdown("**Car share (%)**")); display(share)
display(Markdown("**Revenue (thousand CHF)** (the linear variant only models the car share)")); display(rev)

**Car share (%)**

,Central (net of Sion),CI: strong response,CI: weak response,Before-after,By transit class,Linear LPM (c),Spread (pp)
Fee (CHF),,,,,,,
4,43.9,43.9,43.9,43.9,43.9,43.9,0.0
5,40.4,39.7,41.2,41.1,41.2,39.7,1.5
6,37.8,36.5,39.1,38.9,39.3,36.3,3.0
8,34.0,32.1,36.0,35.7,36.4,30.9,5.5
10,31.3,29.0,33.8,33.4,34.5,26.7,7.8


**Revenue (thousand CHF)** (the linear variant only models the car share)

,Central (net of Sion),CI: strong response,CI: weak response,Before-after,By transit class,Spread (kCHF)
Fee (CHF),,,,,,
4,223,223,223,223,223,0
5,251,244,258,257,259,15
6,276,262,291,289,293,31
8,322,294,351,348,357,63
10,362,322,407,402,418,96


## 3. Charge par groupe selon le tarif

**Modèle** : élasticités par classe d'accès TP (comme la partie E). Chaque automobiliste de 2025 continue à
conduire avec la probabilité (tarif / 4)^ε_ext de sa classe et ajuste ses jours de (tarif / 4)^ε_int.

**Mesure** : dépense annuelle de parking en % du revenu brut observé, **moyenne sur les automobilistes restants**
(pondérée par leur probabilité de continuer), comme la colonne (c) de la Table E1. À 4 CHF, on retrouve la
colonne (b) de E1.

In [5]:
d25 = EL[EL.uses_car == 1].copy()                      # Lausanne drivers in 2025
e_ext = d25.transit_class.map(lambda c: eps_cls[c][0]).astype(float)
e_int = d25.transit_class.map(lambda c: eps_cls[c][1]).astype(float)


def project(fee_by_class):
    # fee_by_class: dict transit class -> daily fee (CHF). Returns one row per 2025 driver.
    f = d25.transit_class.map(fee_by_class).astype(float)
    r = f / FEE_2025
    out = d25[["employee_id", "transit_class", "income_tercile", "children_group", "gross_income_chf"]].copy()
    out["p_stay"] = r ** e_ext
    out["days"] = d25.car_days_per_week * r ** e_int
    out["spend"] = f * out.days * WEEKS
    out["burden"] = 100 * out.spend / out.gross_income_chf
    return out


def uniform(p):
    return {c: p for c in ORDER_T}


# consistency check with the by-class scenario of Part D
for k, p in enumerate(FEES):
    pr = project(uniform(p))
    assert np.isclose((pr.p_stay * pr.spend).sum(), cls.rev.iloc[k])

GROUP_LIST = [("Overall", None, "All")] + [(t, v, lev) for t, v, lv in GROUPS for lev in lv]


def burden_table(policies, what="drivers"):
    rows = {}
    for name, fees in policies.items():
        pr = project(fees)
        col = {}
        for t, v, lev in GROUP_LIST:
            g = pr if v is None else pr[pr[v] == lev]
            if what == "drivers":
                col[(t, lev)] = np.average(g.burden, weights=g.p_stay)
            else:   # all employees of the group, non-drivers count as 0
                n = len(EL) if v is None else (EL[v] == lev).sum()
                col[(t, lev)] = (g.p_stay * g.burden).sum() / n
        rows[name] = col
    out = pd.DataFrame(rows).round(2)
    out.index.names = ["Dimension", "Group"]
    return out


UNIFORM = {f"{p} CHF": uniform(p) for p in FEES}
display(Markdown("**Remaining drivers: parking spending as % of gross income**"))
display(burden_table(UNIFORM))
display(Markdown("**All employees of the group (non-drivers count as 0): % of gross income**"))
display(burden_table(UNIFORM, what="all"))

**Remaining drivers: parking spending as % of gross income**

4 CHF  5 CHF  6 CHF  8 CHF  10 CHF
Dimension            Group                                                 
Overall              All                  0.69   0.85   1.02   1.34    1.66
Transit access       Good                 0.57   0.70   0.82   1.05    1.28
                     Medium               0.70   0.87   1.03   1.35    1.67
                     Poor                 0.80   0.99   1.18   1.55    1.91
Income tercile (FTE) T1 (low)             0.97   1.20   1.43   1.89    2.34
                     T2 (middle)          0.71   0.87   1.04   1.37    1.69
                     T3 (high)            0.50   0.62   0.73   0.96    1.19
Children < 16        0 children           0.56   0.69   0.82   1.08    1.33
                     1 child              0.71   0.88   1.05   1.39    1.73
                     2 or more children   0.83   1.02   1.21   1.60    1.98

**All employees of the group (non-drivers count as 0): % of gross income**

4 CHF  5 CHF  6 CHF  8 CHF  10 CHF
Dimension            Group                                                 
Overall              All                  0.30   0.35   0.40   0.49    0.57
Transit access       Good                 0.17   0.18   0.19   0.21    0.23
                     Medium               0.33   0.38   0.43   0.52    0.60
                     Poor                 0.69   0.85   1.00   1.31    1.60
Income tercile (FTE) T1 (low)             0.35   0.41   0.46   0.57    0.67
                     T2 (middle)          0.30   0.34   0.39   0.48    0.56
                     T3 (high)            0.27   0.31   0.35   0.42    0.49
Children < 16        0 children           0.19   0.22   0.25   0.31    0.36
                     1 child              0.35   0.41   0.47   0.57    0.67
                     2 or more children   0.48   0.56   0.63   0.77    0.90

### Charges élevées : combien d'automobilistes au-dessus de 1 % et 2 % du revenu ?
Parts parmi les automobilistes restants (pondérées par la probabilité de continuer), charge maximale individuelle
et dépense annuelle moyenne.

In [6]:
rows = []
for p in FEES:
    pr = project(uniform(p))
    w = pr.p_stay
    rows.append({"Fee (CHF)": p, "Expected drivers": round(w.sum()),
                 "Mean spending (CHF/year)": round(np.average(pr.spend, weights=w)),
                 "Above 1 % of income (%)": round(100 * np.average(pr.burden > 1, weights=w), 1),
                 "Above 2 % of income (%)": round(100 * np.average(pr.burden > 2, weights=w), 1),
                 "Maximum (%)": round(pr.burden.max(), 2),
                 "Poor access, mean (%)": round(np.average(pr.burden[pr.transit_class == "Poor"],
                                                          weights=w[pr.transit_class == "Poor"]), 2),
                 "Low income T1, mean (%)": round(np.average(pr.burden[pr.income_tercile == "T1 (low)"],
                                                            weights=w[pr.income_tercile == "T1 (low)"]), 2)})
pd.DataFrame(rows)

,Fee (CHF),Expected drivers,Mean spending (CHF/year),Above 1 % of income (%),Above 2 % of income (%),Maximum (%),"Poor access, mean (%)","Low income T1, mean (%)"
0,4,351,635,12.8,0.0,1.86,0.80,0.97
1,5,330,784,27.1,2.0,2.26,0.99,1.20
2,6,314,932,44.1,3.3,2.69,1.18,1.43
3,8,292,1224,69.0,11.1,3.53,1.55,1.89
4,10,276,1514,86.7,25.7,4.36,1.91,2.34


## 4. Qui réduit son usage de la voiture, qui paie ?

Par classe d'accès TP (modèle par classe) : automobilistes attendus, automobilistes en moins par rapport à 2025,
et part de la recette payée par la classe. Rappel : en 2025, la classe Poor compte 134 employés (16.8 %).

In [7]:
rows = []
for p in FEES:
    pr = project(uniform(p))
    tot = (pr.p_stay * pr.spend).sum()
    for c in ORDER_T:
        g = pr[pr.transit_class == c]
        rows.append({"Fee (CHF)": p, "Transit access": c, "Expected drivers": round(g.p_stay.sum()),
                     "Fewer than 2025": round(len(g) - g.p_stay.sum()),
                     "Share of revenue (%)": round(100 * (g.p_stay * g.spend).sum() / tot, 1)})
who = pd.DataFrame(rows).pivot(index="Fee (CHF)", columns="Transit access")
who = who.reindex(columns=ORDER_T, level=1)
who

Expected drivers             Fewer than 2025             Share of revenue (%)             
Transit access             Good Medium Poor            Good Medium Poor                 Good Medium  Poor
Fee (CHF)                                                                                                
4                           123    113  115               0      0    0                 30.8   32.7  36.5
5                           110    106  114              13      7    1                 28.7   32.6  38.7
6                           100    101  114              23     12    1                 27.1   32.5  40.4
8                            86     93  113              37     20    2                 24.6   32.1  43.2
10                           77     87  112              46     26    3                 22.8   31.8  45.4

## 5. Synthèse pour la décision

**Efficacité : rendements décroissants.** Chaque franc retire moins d'automobilistes que le précédent : environ 28
entre 4 et 5 CHF, 21 entre 5 et 6, 15 par franc entre 6 et 8, 11 par franc entre 8 et 10. Le passage de 6 à 8 CHF
retire encore 30 automobilistes (302 → 272) et 25 voitures garées par jour (200 → 175).

**Fiabilité : bonne jusqu'à 6 CHF, plus fragile ensuite.** L'écart entre variantes est de 3.0 pp à 6 CHF, 5.5 pp à
8 CHF et 7.8 pp à 10 CHF. 8 CHF est la limite de la plage crédible ; 10 CHF est une extrapolation.

**Recette : elle augmente toujours** (276'000 CHF à 6, 322'000 à 8), mais cet argument ne doit pas décider seul
(consignes, partie F).

**Charge : le point qui distingue le plus les options.**
- À 6 CHF, 44 % des automobilistes restants dépassent 1 % de leur revenu ; à 8 CHF, 69 %, et 11 % dépassent 2 %.
- Le tercile de revenu bas passe de 0.97 % (2025) à 1.43 % (6 CHF) puis 1.89 % (8 CHF).
- Les employés mal desservis ne partent presque pas (114 sur 115 restent à 6 CHF, 113 à 8 CHF) : pour eux, toute
  hausse est une charge sans effet sur la circulation. Leur part de la recette passe de 36.5 % à 43.2 % à 8 CHF.

**Pistes pour la discussion de groupe** (à vous de trancher) :
1. **6 CHF** : prévision la plus fiable, hausse modérée (+50 %), 49 automobilistes de moins ; la charge des
   automobilistes restants passe à 1.02 % en moyenne (1.43 % pour le tercile bas).
2. **8 CHF** : 30 automobilistes de moins qu'à 6 CHF et 46'000 CHF de recette en plus, mais à la limite de la plage
   crédible et avec une charge nettement plus lourde (69 % des automobilistes restants au-dessus de 1 % du revenu).
3. **5 CHF** : peu d'effet (28 automobilistes de moins) ; **10 CHF** : hors de la plage crédible.
L'option écartée peut servir d'« alternative considérée » dans le mémo.

**Limites de ces comparaisons** : les effets par groupe (sections 3 et 4) utilisent les élasticités par classe
d'accès (avant-après, sans correction par Sion), qui prévoient un peu moins de départs que le scénario central (314
contre 302 automobilistes à 6 CHF). Rien n'est dit sur le bien-être ni sur le coût des alternatives (non observés).